# Stage 40 — Classificação com SVM

Objetivo: classificar as quatro palavras separadamente para cada participante. A padronização, seleção de características e SVM ficam no mesmo `Pipeline`.


## 1. Importações


In [1]:
from pathlib import Path
import pickle
import re

import numpy as np
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC


from config.channel_mapping import INDICES_CHANNELS_MAPPING



## 2. Caminhos e parâmetros do experimento


In [2]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

INPUT_DIR = PROJECT_ROOT / "processed_data" / "stage_02_feature_extraction_psd"
OUTPUT_DIR = PROJECT_ROOT / "processed_data" / "stage_03_svm"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

#K_FEATURES = 3
N_SPLITS = 5
RANDOM_STATE = 42
CLASS_LABELS = [0, 1, 2, 3]


## 3. Descobrir os participantes


In [3]:
feature_files = sorted(INPUT_DIR.glob("sub-*_ses-*_features_psd.npy"))
subjects = sorted({
    re.match(r"(sub-\d+)", path.name).group(1)
    for path in feature_files
})
print("Participantes:", subjects)


Participantes: ['sub-01', 'sub-02', 'sub-03', 'sub-04', 'sub-05', 'sub-06', 'sub-07', 'sub-08', 'sub-09', 'sub-10']


## 4. Função para juntar as sessões

As três sessões de um participante são concatenadas no eixo das épocas.


In [4]:
def load_subject(subject):
    session_files = sorted(INPUT_DIR.glob(f"{subject}_ses-*_features_psd.npy"))
    session_features = []
    session_labels = []

    for feature_path in session_files:

        session_name = feature_path.name.replace("_features_psd.npy", "")
        labels_path = INPUT_DIR / f"{session_name}_labels.npy"

        session_features.append(np.load(feature_path))
        session_labels.append(np.load(labels_path))

    if not session_features:
        raise FileNotFoundError(f"Nenhuma sessão encontrada para {subject}")

    features = np.concatenate(session_features, axis=0)
    labels = np.concatenate(session_labels, axis=0)
    
    return features, labels


## 5. Inspecionar um participante


In [5]:
example_features, example_labels = load_subject(subjects[0])
print("Características:", example_features.shape)
print("Rótulos:", example_labels.shape)
print("Classes:", np.unique(example_labels, return_counts=True))


Características: (200, 128, 4)
Rótulos: (200,)
Classes: (array([0, 1, 2, 3]), array([50, 50, 50, 50]))


## 6. Construir o pipeline

Em cada fold, o scaler e o seletor são ajustados somente no conjunto de treino.


In [ ]:
# model = Pipeline([
#     ("scaler", StandardScaler()),
#     ("selector", SelectKBest(score_func=f_classif, k=K_FEATURES)),
#     ("svm", SVC(kernel="rbf", C=1.0, gamma="scale")),
# ])

# cross_validation = StratifiedKFold(
#     n_splits=N_SPLITS,
#     shuffle=True,
#     random_state=RANDOM_STATE,
# )

# model


## 7. Função de avaliação de um participante


In [7]:
def evaluate_subject(subject, k_features):
    features, labels = load_subject(subject)
    
    n_channels = features.shape[1]
    n_measures = features.shape[2]

    # (épocas, canais, features) -> (épocas, canais * features)
    features = features.reshape(features.shape[0], -1)

    scores = []
    matrices = []
    selected_features_in_all_folds = []

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("selector", SelectKBest(score_func=f_classif, k=k_features)),
        ("svm", SVC(kernel="rbf", C=1.0, gamma="scale")),
    ])

    cross_validation = StratifiedKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE,
    ) 

    for train_index, test_index in cross_validation.split(features, labels):

        x_train = features[train_index]
        x_test = features[test_index]
        y_train = labels[train_index]
        y_test = labels[test_index]

        model.fit(x_train, y_train)

        y_predict = model.predict(x_test)

        scores.append(
            accuracy_score(y_test, y_predict)
        )

        matrices.append(
            confusion_matrix(
                y_test,
                y_predict,
                labels=CLASS_LABELS,
                normalize="true",
            )
        )

        indices = np.flatnonzero(
            model.named_steps["selector"].get_support()
        )

        feature_info = []

        for feature_idx in indices:

            channel_idx, measure_idx = np.unravel_index(
                feature_idx,
                (n_channels, n_measures)
            )

            feature_info.append({
                "feature_idx": int(feature_idx),
                "channel_index": int(channel_idx),
                "channel_name": INDICES_CHANNELS_MAPPING[channel_idx],
                "measure_index": int(measure_idx),
            })

        selected_features_in_all_folds.append(feature_info)

    scores = np.asarray(scores)
    matrices = np.asarray(matrices)

    return {
        "k_features": k_features,
        "scores": scores,
        "mean_accuracy": scores.mean(),
        "std_accuracy": scores.std(),
        "confusion_matrices": matrices,
        "mean_confusion_matrix": matrices.mean(axis=0),
        "selected_features_in_all_folds": selected_features_in_all_folds,
    }

## 8. Avaliar apenas um participante

Use esta célula para entender a saída antes de executar todos.


In [8]:
example_subject = subjects[0]
example_result = evaluate_subject(example_subject, 3)

print("Participante:", example_subject)
print("Acurácias dos folds:", example_result["scores"])
print("Média:", example_result["mean_accuracy"])
print("Matriz média:\n", example_result["mean_confusion_matrix"])


Participante: sub-01
Acurácias dos folds: [0.275 0.225 0.25  0.275 0.25 ]
Média: 0.255
Matriz média:
 [[0.34 0.26 0.22 0.18]
 [0.14 0.36 0.22 0.28]
 [0.22 0.26 0.12 0.4 ]
 [0.22 0.46 0.12 0.2 ]]


## 9. Avaliar e salvar todos os participantes


In [10]:
n_channels = 128
n_measures = 2

for k_features in range(1, 6): 
    for subject in subjects:
        result = evaluate_subject(subject, k_features)
        output_path = OUTPUT_DIR / f"k_features_{k_features}" / f"{subject}_svm_results_.pkl"
        
        # Cria a pasta e subpastas se não existirem
        output_path.parent.mkdir(parents=True, exist_ok=True)
        
        with output_path.open("wb") as file:
            pickle.dump({subject: result}, file)
            
        print(subject, f"acurácia média={result['mean_accuracy']:.3f}")

    print("Stage 03 finalizado.")



sub-01 acurácia média=0.275
sub-02 acurácia média=0.212
sub-03 acurácia média=0.239
sub-04 acurácia média=0.217
sub-05 acurácia média=0.304
sub-06 acurácia média=0.213
sub-07 acurácia média=0.254
sub-08 acurácia média=0.230
sub-09 acurácia média=0.300
sub-10 acurácia média=0.258
Stage 03 finalizado.
sub-01 acurácia média=0.265
sub-02 acurácia média=0.204
sub-03 acurácia média=0.272
sub-04 acurácia média=0.242
sub-05 acurácia média=0.283
sub-06 acurácia média=0.245
sub-07 acurácia média=0.217
sub-08 acurácia média=0.225
sub-09 acurácia média=0.267
sub-10 acurácia média=0.267
Stage 03 finalizado.
sub-01 acurácia média=0.255
sub-02 acurácia média=0.237
sub-03 acurácia média=0.244
sub-04 acurácia média=0.208
sub-05 acurácia média=0.287
sub-06 acurácia média=0.222
sub-07 acurácia média=0.233
sub-08 acurácia média=0.235
sub-09 acurácia média=0.279
sub-10 acurácia média=0.258
Stage 03 finalizado.
sub-01 acurácia média=0.210
sub-02 acurácia média=0.242
sub-03 acurácia média=0.244
sub-04 acurác